# 8.1 Lab: Benchmarking LLM Inference

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/09_operations/08.1_benchmarking/lab.ipynb) [![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.marimo.io/github/harshuljain13/llm-inference-at-scale/blob/master/content/09_operations/08.1_benchmarking/lab.ipynb)

Simulate realistic LLM benchmark workloads, measure TTFT/ITL/E2E distributions,
validate against SLO profiles, and diagnose bottlenecks.

In [ ]:
# --- Setup: install and import required libraries ---
import subprocess, sys
# Ensure numpy and matplotlib are available in this environment
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'numpy', 'matplotlib'])

# Core numerical computing library for statistics and array operations
import numpy as np
# Plotting library for latency distribution histograms and charts
import matplotlib.pyplot as plt
# Dataclass decorator for creating structured metric containers
from dataclasses import dataclass, field
# Type hints for function signatures and documentation
from typing import List, Dict
# Note: no GPU or inference server required for this simulation lab

In [ ]:
# --- RequestMetrics: structured container for one request's measurements ---
@dataclass
class RequestMetrics:
    """Stores all timing measurements for a single LLM inference request."""
    # Time to first token: measures prefill + queue latency
    ttft: float = 0.0
    # List of inter-token latencies: one value per generated token
    tbt: List[float] = field(default_factory=list)
    # End-to-end latency: total time from request send to last token
    e2e: float = 0.0
    # Count of output tokens the model produced
    tokens_generated: int = 0
    # Count of input tokens in the prompt
    input_tokens: int = 0
    # Whether the request completed without errors
    success: bool = True

    @property
    def mean_tbt(self) -> float:
        # Average inter-token latency across all generated tokens
        return float(np.mean(self.tbt)) if self.tbt else 0.0

    @property
    def p99_tbt(self) -> float:
        # 99th percentile inter-token latency (tail behavior)
        return float(np.percentile(self.tbt, 99)) if self.tbt else 0.0

In [ ]:
# --- WorkloadGenerator: create realistic traffic patterns ---
class WorkloadGenerator:
    """Generates synthetic prompts and arrival patterns matching production."""

    # Traffic profiles define input length range and output target per use case
    PROFILES = {
        # Chatbot: short prompts, moderate output
        "chatbot": {"input_range": (20, 200), "output_tokens": 128},
        # Code completion: longer context, long output
        "code": {"input_range": (100, 2000), "output_tokens": 512},
        # Batch processing: very long inputs, moderate output
        "batch": {"input_range": (500, 4000), "output_tokens": 256},
        # Voice/realtime: ultra-short prompts, minimal output
        "voice": {"input_range": (5, 50), "output_tokens": 64},
    }

    def __init__(self, profile: str = "chatbot", seed: int = 42):
        # Select the traffic profile configuration
        self.profile = self.PROFILES[profile]
        # Use fixed seed for reproducible benchmark runs
        self.rng = np.random.default_rng(seed)

    def generate_prompts(self, n: int) -> List[int]:
        """Return n input token lengths from the profile's distribution."""
        lo, hi = self.profile["input_range"]
        # Uniform distribution within the profile's range
        return self.rng.integers(lo, hi, size=n).tolist()

    def poisson_arrivals(self, n: int, rps: float) -> List[float]:
        """Generate Poisson arrival timestamps at the specified rate."""
        # Exponential inter-arrival times model memoryless real traffic
        intervals = self.rng.exponential(1.0 / rps, size=n)
        # Cumulative sum converts intervals to absolute timestamps
        return np.cumsum(intervals).tolist()


# --- Demo: show what the workload generator produces ---
wg = WorkloadGenerator("chatbot")
# Generate 10 prompt lengths following chatbot distribution
# Generate 10 synthetic prompt lengths from chatbot distribution
lengths = wg.generate_prompts(10)
# Generate 10 Poisson arrival times at 5 requests per second
# Simulate 10 request arrivals following Poisson process at 5 req/s
arrivals = wg.poisson_arrivals(10, rps=5.0)
print(f"Prompt lengths (tokens): {lengths}")
print(f"Arrival times (seconds): {[f'{t:.2f}' for t in arrivals]}")

In [ ]:
# --- Simulate 500 benchmark results with realistic latency distributions ---
# In production, replace with actual async streaming HTTP client measurements
def simulate_benchmark(n_requests: int = 500, seed: int = 42) -> List[RequestMetrics]:
    """Generate synthetic but statistically realistic benchmark results."""
    rng = np.random.default_rng(seed)
    # Accumulate results for all simulated requests
    results = []

    for _ in range(n_requests):
        # TTFT: log-normal with median ~80ms (matches real vLLM measurements)
        # Long tail from queue effects and variable prompt lengths
        ttft = rng.lognormal(np.log(0.08), 0.4)
        # Each request generates a variable number of output tokens
        n_tokens = rng.integers(30, 200)
        # ITL: log-normal with median ~25ms per token
        # Spikes occur when batch size fluctuates (bandwidth contention)
        tbt = rng.lognormal(np.log(0.025), 0.3, size=n_tokens).tolist()
        # Total time = prefill (TTFT) + sum of all decode steps
        e2e = ttft + sum(tbt)
        # Package all measurements into structured container
        results.append(RequestMetrics(
            ttft=ttft, tbt=tbt, e2e=e2e,
            tokens_generated=n_tokens,
            input_tokens=rng.integers(20, 500)
        ))

    return results

# Execute the simulation and print summary statistics
sim_results = simulate_benchmark(500)
print(f"Simulated {len(sim_results)} requests")
# Compute median TTFT across all requests (in milliseconds)
print(f"TTFT median: {np.median([r.ttft*1000 for r in sim_results]):.1f} ms")
# Compute median ITL across all requests (in milliseconds)
print(f"ITL median: {np.median([r.mean_tbt*1000 for r in sim_results]):.1f} ms")

In [ ]:
# --- Visualize latency distributions with percentile markers ---
def plot_latency_distributions(results: List[RequestMetrics]):
    """Three-panel histogram: TTFT, ITL, E2E with P50/P95/P99 lines."""
    # Create side-by-side panels for the three key metrics
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # Convert all latencies from seconds to milliseconds for readability
    ttfts = [r.ttft * 1000 for r in results]
    itls = [r.mean_tbt * 1000 for r in results if r.tbt]
    e2es = [r.e2e * 1000 for r in results]

    # Plot histogram + percentile lines for each metric
    for ax, data, title in zip(axes, [ttfts, itls, e2es],
                                ["TTFT (ms)", "Mean ITL (ms)", "E2E (ms)"]):
        # Draw histogram of latency values
        ax.hist(data, bins=50, alpha=0.7, color='steelblue', edgecolor='black', lw=0.5)
        # Overlay vertical lines at P50 (green), P95 (orange), P99 (red)
        for pct, color in [(50, 'green'), (95, 'orange'), (99, 'red')]:
            val = np.percentile(data, pct)
            # Dashed line with legend showing the actual value
            ax.axvline(val, color=color, ls='--', label=f'P{pct}: {val:.0f}ms')
        ax.set_title(title, fontsize=11)
        ax.set_xlabel('Latency (ms)')
        ax.legend(fontsize=8)

    plt.tight_layout()
    plt.show()

# Render the distribution plots for our simulated benchmark
plot_latency_distributions(sim_results)

In [ ]:
# --- SLO Validation: check if benchmark meets service level objectives ---
@dataclass
class SLO:
    """Defines latency limits for a specific use case."""
    # Human-readable profile name
    name: str
    # Maximum P99 time-to-first-token in milliseconds
    ttft_p99_ms: float
    # Maximum P99 inter-token-latency in milliseconds
    itl_p99_ms: float
    # Maximum P99 end-to-end latency in milliseconds
    e2e_p99_ms: float

# Define SLO targets for each production use case
SLOS = {
    # Voice requires ultra-low latency for real-time conversation
    "voice": SLO("voice", ttft_p99_ms=150, itl_p99_ms=50, e2e_p99_ms=2000),
    # Chatbot needs fast first token for perceived responsiveness
    "chatbot": SLO("chatbot", ttft_p99_ms=300, itl_p99_ms=80, e2e_p99_ms=5000),
    # Code completion tolerates higher TTFT but needs consistent ITL
    "code": SLO("code", ttft_p99_ms=1000, itl_p99_ms=100, e2e_p99_ms=30000),
    # Batch is latency-tolerant, optimized for throughput
    "batch": SLO("batch", ttft_p99_ms=5000, itl_p99_ms=200, e2e_p99_ms=120000),
}

def validate_slo(results: List[RequestMetrics], slo: SLO) -> Dict:
    """Compare actual P99 values against SLO limits."""
    # Compute actual P99 TTFT from benchmark measurements
    ttft_p99 = np.percentile([r.ttft * 1000 for r in results], 99)
    # Compute actual P99 ITL from per-request P99 values
    itl_p99 = np.percentile([r.p99_tbt * 1000 for r in results if r.tbt], 99)
    # Compute actual P99 E2E latency
    e2e_p99 = np.percentile([r.e2e * 1000 for r in results], 99)
    # Return structured pass/fail report
    return {
        "ttft": {"actual": ttft_p99, "limit": slo.ttft_p99_ms, "pass": ttft_p99 <= slo.ttft_p99_ms},
        "itl": {"actual": itl_p99, "limit": slo.itl_p99_ms, "pass": itl_p99 <= slo.itl_p99_ms},
        "e2e": {"actual": e2e_p99, "limit": slo.e2e_p99_ms, "pass": e2e_p99 <= slo.e2e_p99_ms},
    }

# Validate our benchmark against every SLO profile
# Iterate through all profiles and report pass/fail for each
for name, slo in SLOS.items():
    result = validate_slo(sim_results, slo)
    # Determine overall pass/fail for this profile
    all_pass = all(v['pass'] for v in result.values())
    icon = '\u2705' if all_pass else '\u274c'
    print(f"{icon} {name.upper()}:")
    # Print each metric's actual vs limit
    for metric, v in result.items():
        flag = '\u2713' if v['pass'] else '\u2717'
        print(f"   {flag} {metric}: {v['actual']:.0f} / {v['limit']:.0f} ms")
    print()

In [ ]:
# --- Bottleneck Diagnosis: classify the primary performance limiter ---
def diagnose_bottleneck(results: List[RequestMetrics]) -> Dict:
    """Determine if system is prefill-bound, decode-bound, or schedule-bound."""
    # Filter to successful requests that have decode data
    successful = [r for r in results if r.success and r.tbt]

    # Extract timing arrays for statistical analysis
    ttfts = np.array([r.ttft for r in successful])
    decode_times = np.array([sum(r.tbt) for r in successful])
    e2es = np.array([r.e2e for r in successful])

    # Calculate what fraction of total time each phase consumes
    # Prefill fraction: proportion of E2E spent waiting for first token
    prefill_frac = np.mean(ttfts / e2es)
    # Decode fraction: proportion of E2E spent generating tokens
    decode_frac = np.mean(decode_times / e2es)
    # Overhead: scheduling, queuing, network (everything not prefill or decode)
    overhead_frac = 1 - prefill_frac - decode_frac

    # Coefficient of variation shows which phase has most variability
    # High CV = high variance relative to mean = likely bottleneck
    ttft_cv = np.std(ttfts) / np.mean(ttfts)
    decode_cv = np.std(decode_times) / np.mean(decode_times)

    # Classification rules based on empirical thresholds
    if prefill_frac > 0.4 or ttft_cv > decode_cv * 1.5:
        # Prefill dominates: long prompts or compute saturation
        bottleneck = "PREFILL-BOUND"
        fix = "chunked prefill, prefix caching, tensor parallelism"
    elif overhead_frac > 0.2:
        # Scheduling overhead dominates: queue buildup
        bottleneck = "SCHEDULE-BOUND"
        fix = "reduce max_num_seqs, add replicas"
    else:
        # Decode dominates: memory bandwidth is the limit
        bottleneck = "DECODE-BOUND"
        fix = "quantization, speculative decoding, GQA models"

    return {"bottleneck": bottleneck, "fix": fix,
            "prefill_pct": f"{prefill_frac:.1%}",
            "decode_pct": f"{decode_frac:.1%}",
            "overhead_pct": f"{overhead_frac:.1%}"}

# Run diagnosis and print recommendation
diag = diagnose_bottleneck(sim_results)
print(f"Diagnosis: {diag['bottleneck']}")
print(f"Recommended fix: {diag['fix']}")
print(f"Time breakdown: prefill={diag['prefill_pct']} decode={diag['decode_pct']} overhead={diag['overhead_pct']}")

In [ ]:
# --- Saturation Sweep: find the capacity knee where SLO breaks ---
def simulate_saturation_sweep() -> Dict:
    """Model how latency degrades as request rate approaches capacity."""
    rng = np.random.default_rng(99)
    # Test increasing request rates from idle to overloaded
    rates = [1, 2, 5, 10, 20, 50, 100, 200]
    # Track P99 TTFT at each load level
    ttft_p99s = []
    # Track achieved throughput (saturates at max capacity)
    throughputs = []

    for rps in rates:
        # Load factor: fraction of maximum capacity being used
        load_factor = rps / 150.0  # assume 150 rps = max capacity
        # M/M/1 queue model: wait time grows exponentially near saturation
        queue_multiplier = 1.0 / max(0.01, 1.0 - load_factor)
        # TTFT P99 = base latency * queue effect + noise
        base_ttft = 80  # 80ms at zero load (pure prefill time)
        ttft_p99 = base_ttft * queue_multiplier + rng.normal(0, 5)
        # Floor at base TTFT (can't be faster than compute allows)
        ttft_p99s.append(max(base_ttft, ttft_p99))
        # Throughput = min(offered, capacity) * avg output length
        throughputs.append(min(rps, 150) * 128)

    return {"rates": rates, "ttft_p99s": ttft_p99s, "throughputs": throughputs}

# Generate and plot the saturation curves
sweep = simulate_saturation_sweep()
fig_c7, (ax1_c7, ax2_c7) = plt.subplots(1, 2, figsize=(12, 4))

# Left panel: TTFT P99 vs request rate (find the capacity knee)
ax1_c7.semilogy(sweep['rates'], sweep['ttft_p99s'], 'b-o', lw=2)
# Red dashed line shows where the chatbot SLO limit sits
ax1_c7.axhline(300, color='red', ls='--', label='Chatbot SLO (300ms)')
ax1_c7.set_xlabel('Request Rate (req/s)')
ax1_c7.set_ylabel('TTFT P99 (ms, log scale)')
ax1_c7.set_title('Saturation Sweep: Latency vs Load')
ax1_c7.legend()
ax1_c7.grid(True, alpha=0.3)

# Right panel: throughput vs offered load (shows ceiling)
ax2_c7.plot(sweep['rates'], [t/1000 for t in sweep['throughputs']], 'g-o', lw=2)
ax2_c7.set_xlabel('Request Rate (req/s)')
ax2_c7.set_ylabel('Throughput (K tokens/s)')
ax2_c7.set_title('Throughput Saturation Ceiling')
ax2_c7.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()